# Entrenamiento y comparación experimental — Unidad 4

**SafeRest 360 · Vigía de zonas.** La red responde una sola pregunta: **¿hay al menos una persona en la imagen de la zona?** (`ocupada` = 1, `vacía` = 0).

- **Dataset:** subconjunto al estilo *Visual Wake Words* (Chowdhery et al., 2019) construido desde **COCO val2017** con la regla oficial de VWW: la imagen es `ocupada` si tiene al menos una persona cuya área supera el 0,5 % del área de la imagen.
- **Preprocesamiento:** OpenCV (`src/vision.py`), el mismo que usan el agente y la app.
- **Configuraciones comparadas:** (1) CNN propia entrenada desde cero — *modelo base*; (2) MobileNetV2 preentrenada en ImageNet + cabeza con Dropout — *modelo mejorado*.

> Ejecuten el notebook completo (**Kernel → Restart & Run All**) y guárdenlo **con las salidas**: el README solo puede citar cifras que aparezcan aquí o en `assets/`.

## 1. Configuración

In [ ]:
# Si trabajan en Google Colab, descomenten estas dos líneas y pongan la URL de SU repositorio:
# !git clone https://github.com/USUARIO/saferest360-vigia-zonas.git
# %cd saferest360-vigia-zonas

In [ ]:
import json
import random
import shutil
import sys
import time
import urllib.request
import zipfile
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src.config import (DIR_ASSETS, DIR_DATOS, DIR_EJEMPLOS, DIR_FALLOS, DIR_MODELOS,
                        RUTA_INFO_MODELO, TAMANO_IMAGEN, UMBRAL_DECISION)
from src.vision import brillo_medio, leer_imagen_bgr, preprocesar

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)

for carpeta in (DIR_DATOS, DIR_MODELOS, DIR_ASSETS, DIR_FALLOS, DIR_EJEMPLOS):
    carpeta.mkdir(parents=True, exist_ok=True)

NOMBRES = {0: "vacía", 1: "ocupada"}
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| OpenCV", cv2.__version__)
print("GPU disponibles:", tf.config.list_physical_devices("GPU"))
print("Raíz del proyecto:", RAIZ)

## 2. Datos: COCO val2017 (5.000 imágenes) + anotaciones

In [ ]:
DIR_COCO = DIR_DATOS / "coco"
DIR_COCO.mkdir(parents=True, exist_ok=True)
URL_IMAGENES = "http://images.cocodataset.org/zips/val2017.zip"                    # ~778 MB
URL_ANOTACIONES = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"  # ~241 MB


def descargar(url, destino):
    if destino.exists():
        print("Ya descargado:", destino.name)
        return
    print("Descargando", url)
    inicio = time.time()
    urllib.request.urlretrieve(url, destino)
    print(f"  {destino.stat().st_size / 1e6:.0f} MB en {time.time() - inicio:.0f} s")


zip_imagenes = DIR_COCO / "val2017.zip"
zip_anotaciones = DIR_COCO / "annotations_trainval2017.zip"
descargar(URL_IMAGENES, zip_imagenes)
descargar(URL_ANOTACIONES, zip_anotaciones)

DIR_IMAGENES = DIR_COCO / "val2017"
if not DIR_IMAGENES.exists():
    with zipfile.ZipFile(zip_imagenes) as z:
        z.extractall(DIR_COCO)
RUTA_ANOTACIONES = DIR_COCO / "annotations" / "instances_val2017.json"
if not RUTA_ANOTACIONES.exists():
    with zipfile.ZipFile(zip_anotaciones) as z:
        z.extract("annotations/instances_val2017.json", DIR_COCO)

print("Imágenes disponibles:", len(list(DIR_IMAGENES.glob("*.jpg"))))

In [ ]:
# Etiquetas con la regla de Visual Wake Words: 1 si alguna persona ocupa más del 0,5 % de la imagen
UMBRAL_AREA_VWW = 0.005

coco = json.loads(RUTA_ANOTACIONES.read_text(encoding="utf-8"))
id_persona = next(c["id"] for c in coco["categories"] if c["name"] == "person")
licencias = {lic["id"]: lic for lic in coco["licenses"]}

area_max_persona = {}
for anotacion in coco["annotations"]:
    if anotacion["category_id"] == id_persona:
        img_id = anotacion["image_id"]
        area_max_persona[img_id] = max(area_max_persona.get(img_id, 0.0), anotacion["area"])

filas = []
for img in coco["images"]:
    fraccion = area_max_persona.get(img["id"], 0.0) / (img["width"] * img["height"])
    filas.append({
        "archivo": img["file_name"],
        "fraccion_persona": fraccion,
        "etiqueta": int(fraccion > UMBRAL_AREA_VWW),
        "licencia": licencias[img["license"]]["name"],
        "url_licencia": licencias[img["license"]]["url"],
        "flickr_url": img.get("flickr_url", ""),
    })
df = pd.DataFrame(filas)

print("Total de imágenes:", len(df))
print(df["etiqueta"].map(NOMBRES).value_counts())
pequenas = int(((df["fraccion_persona"] > 0) & (df["etiqueta"] == 0)).sum())
print(f"Imágenes con personas de 0,5 % o menos del área (la regla VWW las etiqueta 'vacía'): {pequenas}")

In [ ]:
# División estratificada 70 / 15 / 15 por imagen (ninguna imagen aparece en dos conjuntos)
df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=SEMILLA)
df_val, df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp["etiqueta"], random_state=SEMILLA)
df_train, df_val, df_test = (d.reset_index(drop=True) for d in (df_train, df_val, df_test))

assert not set(df_train.archivo) & set(df_val.archivo)
assert not set(df_train.archivo) & set(df_test.archivo)
assert not set(df_val.archivo) & set(df_test.archivo)
for nombre, d in [("entrenamiento", df_train), ("validación", df_val), ("prueba", df_test)]:
    print(f"{nombre:13s}: {len(d):5d} imágenes | proporción 'ocupada' = {d['etiqueta'].mean():.3f}")

## 3. Preprocesamiento con OpenCV

In [ ]:
def cargar_conjunto(d):
    X = np.empty((len(d), TAMANO_IMAGEN, TAMANO_IMAGEN, 3), dtype=np.uint8)
    brillos = np.empty(len(d), dtype=np.float32)
    for i, archivo in enumerate(d["archivo"]):
        bgr = leer_imagen_bgr(DIR_IMAGENES / archivo)   # cv2.imread
        X[i] = preprocesar(bgr)                          # BGR->RGB + letterbox a 160x160
        brillos[i] = brillo_medio(bgr)                   # canal V de HSV
    return X, d["etiqueta"].to_numpy().astype("float32"), brillos


inicio = time.time()
X_train, y_train, _ = cargar_conjunto(df_train)
X_val, y_val, _ = cargar_conjunto(df_val)
X_test, y_test, brillo_test = cargar_conjunto(df_test)
print(f"Preprocesamiento: {time.time() - inicio:.0f} s")
print("X_train", X_train.shape, "| X_val", X_val.shape, "| X_test", X_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 6.4))
for ax, i in zip(axes.flat, np.random.default_rng(SEMILLA).choice(len(X_train), 12, replace=False)):
    ax.imshow(X_train[i])
    ax.set_title(f"{NOMBRES[int(y_train[i])]} · {df_train.archivo[i][-10:-4]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Imágenes de entrenamiento después del preprocesamiento con OpenCV")
plt.tight_layout()
plt.savefig(DIR_ASSETS / "ejemplos_dataset.png", dpi=120)
plt.show()

In [ ]:
TAMANO_LOTE = 32
EPOCAS = 15


def dataset(X, y=None, barajar=False):
    # tf.data convierte cada lote a float32 sin duplicar todo el arreglo en memoria
    ds = tf.data.Dataset.from_tensor_slices((X, y) if y is not None else X)
    if barajar:
        ds = ds.shuffle(len(X), seed=SEMILLA)
    ds = ds.batch(TAMANO_LOTE)
    if y is not None:
        ds = ds.map(lambda a, b: (tf.cast(a, tf.float32), b))
    else:
        ds = ds.map(lambda a: tf.cast(a, tf.float32))
    return ds.prefetch(tf.data.AUTOTUNE)


def probabilidades(modelo, X):
    return modelo.predict(dataset(X), verbose=0).ravel()


def metricas(y, p, umbral=UMBRAL_DECISION):
    y_pred = (p >= umbral).astype(int)
    return {"accuracy": accuracy_score(y, y_pred), "precision": precision_score(y, y_pred, zero_division=0),
            "recall": recall_score(y, y_pred, zero_division=0), "f1": f1_score(y, y_pred, zero_division=0)}


def metricas_keras():
    # se crean instancias nuevas para cada modelo (no se comparten métricas con estado)
    return ["accuracy", keras.metrics.Recall(name="recall"), keras.metrics.Precision(name="precision")]

## 4. Configuración 1 — CNN propia (modelo base)

In [ ]:
def construir_cnn_base():
    modelo = keras.Sequential([
        layers.Input(shape=(TAMANO_IMAGEN, TAMANO_IMAGEN, 3)),
        layers.Rescaling(1.0 / 255),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="cnn_base")
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=metricas_keras())
    return modelo


cnn_base = construir_cnn_base()
cnn_base.summary()

inicio = time.time()
historial_base = cnn_base.fit(dataset(X_train, y_train, barajar=True), validation_data=dataset(X_val, y_val),
                              epochs=EPOCAS, verbose=2)
print(f"Tiempo de entrenamiento (CNN base): {time.time() - inicio:.0f} s")

## 5. Configuración 2 — MobileNetV2 preentrenada + Dropout (modelo mejorado)

In [ ]:
base_mobilenet = keras.applications.MobileNetV2(
    input_shape=(TAMANO_IMAGEN, TAMANO_IMAGEN, 3), include_top=False, weights="imagenet", pooling="avg"
)
base_mobilenet.trainable = False   # se congelan los pesos de ImageNet

entrada = keras.Input(shape=(TAMANO_IMAGEN, TAMANO_IMAGEN, 3))
x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(entrada)      # equivale a mobilenet_v2.preprocess_input
caracteristicas = base_mobilenet(x, training=False)
extractor = keras.Model(entrada, caracteristicas, name="extractor_mobilenetv2")

# Como la base está congelada, las características se calculan una sola vez
inicio = time.time()
F_train = extractor.predict(dataset(X_train), verbose=0)
F_val = extractor.predict(dataset(X_val), verbose=0)
F_test = extractor.predict(dataset(X_test), verbose=0)
print(f"Extracción de características: {time.time() - inicio:.0f} s | forma: {F_train.shape}")

cabeza = keras.Sequential([
    layers.Input(shape=(F_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="cabeza_clasificacion")
cabeza.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=metricas_keras())

inicio = time.time()
historial_mejorado = cabeza.fit(F_train, y_train, validation_data=(F_val, y_val),
                                epochs=EPOCAS, batch_size=TAMANO_LOTE, verbose=2)
print(f"Tiempo de entrenamiento (cabeza MobileNetV2): {time.time() - inicio:.0f} s")

# Modelo completo para producción: imagen uint8 160x160 -> probabilidad de persona
modelo_mejorado = keras.Model(entrada, cabeza(caracteristicas), name="mobilenetv2_transfer")
diferencia = np.abs(cabeza.predict(F_test, verbose=0).ravel() - probabilidades(modelo_mejorado, X_test)).max()
print(f"Comprobación: diferencia máxima entre cabeza y modelo completo = {diferencia:.2e}")

## 6. Curvas de entrenamiento vs. validación

In [ ]:
def graficar_curvas(historial, titulo, archivo):
    h = historial.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, clave, nombre in [(axes[0], "loss", "Pérdida (binary cross-entropy)"), (axes[1], "accuracy", "Exactitud")]:
        ax.plot(range(1, len(h[clave]) + 1), h[clave], marker="o", label="Entrenamiento")
        ax.plot(range(1, len(h["val_" + clave]) + 1), h["val_" + clave], marker="o", label="Validación")
        ax.set_xlabel("Época")
        ax.set_ylabel(nombre)
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.savefig(DIR_ASSETS / archivo, dpi=120)
    plt.show()


graficar_curvas(historial_base, "Configuración 1 · CNN propia (base)", "curvas_cnn_base.png")
graficar_curvas(historial_mejorado, "Configuración 2 · MobileNetV2 + Dropout (mejorado)", "curvas_mobilenetv2.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for h, nombre in [(historial_base.history, "CNN base"), (historial_mejorado.history, "MobileNetV2 + Dropout")]:
    axes[0].plot(range(1, EPOCAS + 1), h["val_loss"], marker="o", label=nombre)
    axes[1].plot(range(1, EPOCAS + 1), h["val_accuracy"], marker="o", label=nombre)
axes[0].set_title("Pérdida en validación")
axes[1].set_title("Exactitud en validación")
for ax in axes:
    ax.set_xlabel("Época")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(DIR_ASSETS / "comparacion_validacion.png", dpi=120)
plt.show()

resumen_epocas = pd.DataFrame({
    nombre: {
        "acc_entrenamiento_final": h["accuracy"][-1], "acc_validacion_final": h["val_accuracy"][-1],
        "loss_entrenamiento_final": h["loss"][-1], "loss_validacion_final": h["val_loss"][-1],
        "mejor_epoca_val_loss": int(np.argmin(h["val_loss"]) + 1),
        "brecha_acc_final (entr - val)": h["accuracy"][-1] - h["val_accuracy"][-1],
    }
    for nombre, h in [("cnn_base", historial_base.history), ("mobilenetv2_transfer", historial_mejorado.history)]
}).T
print(resumen_epocas.round(4).to_string())

## 7. Evaluación: validación (para elegir) y prueba (para reportar)

In [ ]:
p_val = {"cnn_base": probabilidades(cnn_base, X_val), "mobilenetv2_transfer": probabilidades(modelo_mejorado, X_val)}
p_test = {"cnn_base": probabilidades(cnn_base, X_test), "mobilenetv2_transfer": probabilidades(modelo_mejorado, X_test)}

filas_tabla = []
for nombre in p_val:
    fila = {"modelo": nombre}
    for conjunto, y_real, probs in [("val", y_val, p_val), ("test", y_test, p_test)]:
        for clave, valor in metricas(y_real, probs[nombre]).items():
            fila[f"{conjunto}_{clave}"] = valor
    filas_tabla.append(fila)
tabla = pd.DataFrame(filas_tabla).set_index("modelo")
print(tabla.round(4).T.to_string())

# La elección se hace con VALIDACIÓN; el conjunto de prueba solo se usa para reportar
elegido = max(p_val, key=lambda m: metricas(y_val, p_val[m])["accuracy"])
print(f"\nModelo elegido por exactitud en validación: {elegido}")
print("\nReporte de clasificación en PRUEBA del modelo elegido:")
print(classification_report(y_test, (p_test[elegido] >= UMBRAL_DECISION).astype(int),
                            target_names=["vacía", "ocupada"], digits=4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, m in zip(axes, p_test):
    ConfusionMatrixDisplay.from_predictions(y_test, (p_test[m] >= UMBRAL_DECISION).astype(int),
                                            display_labels=["vacía", "ocupada"], cmap="Blues",
                                            colorbar=False, ax=ax)
    ax.set_title(f"{m} · prueba")
plt.tight_layout()
plt.savefig(DIR_ASSETS / "matrices_confusion.png", dpi=120)
plt.show()

## 8. Exportación de los pesos a `models/`

In [ ]:
modelos = {"cnn_base": cnn_base, "mobilenetv2_transfer": modelo_mejorado}
for nombre, modelo in modelos.items():
    ruta = DIR_MODELOS / f"{nombre}.keras"
    modelo.save(ruta)
    print(f"{ruta.name}: {ruta.stat().st_size / 1e6:.1f} MB")   # debe ser < 100 MB para GitHub

info = {
    "archivo_modelo": f"{elegido}.keras",
    "configuracion": elegido,
    "criterio_seleccion": "mayor exactitud en validación",
    "tamano_imagen": TAMANO_IMAGEN,
    "umbral": UMBRAL_DECISION,
    "clases": {"0": "vacía", "1": "ocupada"},
    "metricas_validacion": {k: round(float(v), 4) for k, v in metricas(y_val, p_val[elegido]).items()},
    "metricas_prueba": {k: round(float(v), 4) for k, v in metricas(y_test, p_test[elegido]).items()},
    "dataset": {"origen": "COCO val2017 con regla Visual Wake Words (persona > 0,5 % del área)",
                "entrenamiento": len(df_train), "validacion": len(df_val), "prueba": len(df_test)},
    "semilla": SEMILLA,
    "epocas": EPOCAS,
    "fecha": datetime.now().isoformat(timespec="seconds"),
}
RUTA_INFO_MODELO.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(info, ensure_ascii=False, indent=2))

# Comprobación: el archivo guardado predice igual que el modelo en memoria
recargado = keras.models.load_model(DIR_MODELOS / info["archivo_modelo"])
diferencia = np.abs(probabilidades(recargado, X_test[:64]) - p_test[elegido][:64]).max()
print(f"Diferencia máxima tras recargar el .keras: {diferencia:.2e}")

## 9. Análisis de fallos reales (Error Analysis)

In [ ]:
res = df_test.copy()
res["p_persona"] = p_test[elegido]
res["prediccion"] = (res["p_persona"] >= UMBRAL_DECISION).astype(int)
res["confianza"] = np.where(res["prediccion"] == 1, res["p_persona"], 1 - res["p_persona"])
res["brillo"] = brillo_test
res["correcto"] = res["prediccion"] == res["etiqueta"]


def imprimir_caso(indice, fila):
    print(f"índice={indice} | archivo={fila.archivo} | real={NOMBRES[fila.etiqueta]} | "
          f"predicción={NOMBRES[fila.prediccion]} | p_persona={fila.p_persona:.4f} | "
          f"confianza={fila.confianza:.4f} | fracción_persona={fila.fraccion_persona:.4f} | brillo={fila.brillo:.0f}")


errores = res[~res["correcto"]].sort_values("confianza", ascending=False)
dudosos = res[res["confianza"] < 0.65].sort_values("confianza")
print(f"Modelo analizado: {elegido}")
print(f"Errores en prueba: {len(errores)} de {len(res)} | casos con confianza < 0.65: {len(dudosos)}")
print(f"  falsos 'vacía' (había persona y no la vio): {int(((res.etiqueta == 1) & (res.prediccion == 0)).sum())}")
print(f"  falsos 'ocupada' (vio persona donde no la había): {int(((res.etiqueta == 0) & (res.prediccion == 1)).sum())}")

print("\n== 5 errores en los que la red estaba MÁS segura ==")
for indice, fila in errores.head(5).iterrows():
    imprimir_caso(indice, fila)
print("\n== 5 casos con MENOR confianza ==")
for indice, fila in dudosos.head(5).iterrows():
    imprimir_caso(indice, fila)

In [ ]:
# Se copian las imágenes de los fallos a assets/ para analizarlas en el README y probarlas en la app
for carpeta in (DIR_FALLOS, DIR_EJEMPLOS):
    for viejo in carpeta.glob("*.jpg"):
        viejo.unlink()

seleccion = [("fallo", i, f) for i, f in errores.head(3).iterrows()] + \
            [("dudoso", i, f) for i, f in dudosos.head(2).iterrows() if i not in errores.head(3).index]
creditos = ["# Créditos de las imágenes (COCO val2017)", "",
            "| Archivo en assets | Archivo COCO | Licencia | Enlace de la licencia | Origen |", "|---|---|---|---|---|"]
for k, (tipo, indice, fila) in enumerate(seleccion, start=1):
    destino = DIR_FALLOS / f"{k:02d}_{tipo}_{fila.archivo}"
    shutil.copy(DIR_IMAGENES / fila.archivo, destino)
    creditos.append(f"| fallos/{destino.name} | {fila.archivo} | {fila.licencia} | {fila.url_licencia} | {fila.flickr_url} |")

correctos = res[res["correcto"]].sort_values("confianza", ascending=False)
for clase in (1, 0):
    candidatos = correctos[correctos["etiqueta"] == clase]
    if candidatos.empty:
        print(f"No hay aciertos de la clase '{NOMBRES[clase]}' para guardar como ejemplo")
        continue
    fila = candidatos.iloc[0]
    nombre_clase = "ocupada" if clase == 1 else "vacia"
    destino = DIR_EJEMPLOS / f"ejemplo_{nombre_clase}_{fila.archivo}"
    shutil.copy(DIR_IMAGENES / fila.archivo, destino)
    creditos.append(f"| ejemplos/{destino.name} | {fila.archivo} | {fila.licencia} | {fila.url_licencia} | {fila.flickr_url} |")
(DIR_ASSETS / "CREDITOS_IMAGENES.md").write_text("\n".join(creditos) + "\n", encoding="utf-8")
print("\n".join(creditos))

fig, axes = plt.subplots(1, len(seleccion), figsize=(3.6 * len(seleccion), 4))
for ax, (tipo, indice, fila) in zip(np.atleast_1d(axes), seleccion):
    ax.imshow(cv2.cvtColor(cv2.imread(str(DIR_IMAGENES / fila.archivo)), cv2.COLOR_BGR2RGB))
    ax.set_title(f"[{indice}] real={NOMBRES[fila.etiqueta]}\npred={NOMBRES[fila.prediccion]} p={fila.p_persona:.2f}",
                 fontsize=9)
    ax.axis("off")
plt.suptitle(f"Fallos y casos dudosos en prueba · {elegido}")
plt.tight_layout()
plt.savefig(DIR_ASSETS / "analisis_fallos.png", dpi=120)
plt.show()

In [ ]:
# Desempeño por condición: tamaño de la persona y brillo de la imagen (relevante para restaurantes)
res["tamano_persona"] = pd.cut(res["fraccion_persona"], bins=[-0.001, 0.0, 0.005, 0.02, 0.10, 1.0],
                               labels=["sin persona", "≤0,5 % (etiqueta vacía)", "0,5–2 %", "2–10 %", ">10 %"])
res["iluminacion"] = pd.qcut(res["brillo"], 3, labels=["oscuras", "medias", "claras"])

por_tamano = res.groupby("tamano_persona", observed=False)["correcto"].agg(imagenes="size", exactitud="mean")
por_brillo = res.groupby("iluminacion", observed=False).agg(
    imagenes=("correcto", "size"), exactitud=("correcto", "mean"),
    brillo_min=("brillo", "min"), brillo_max=("brillo", "max"))
print("Exactitud en prueba según el tamaño de la persona más grande:")
print(por_tamano.round(4).to_string())
print("\nExactitud en prueba según el brillo medio (canal V de HSV):")
print(por_brillo.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
por_tamano["exactitud"].plot.bar(ax=axes[0], color="#4E79A7")
axes[0].set_title("Exactitud por tamaño de la persona")
por_brillo["exactitud"].plot.bar(ax=axes[1], color="#F28E2B")
axes[1].set_title("Exactitud por iluminación (terciles)")
for ax in axes:
    ax.set_ylim(0, 1)
    ax.set_ylabel("Exactitud")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(DIR_ASSETS / "desempeno_por_condicion.png", dpi=120)
plt.show()

## 10. Resumen de cifras (para citar en el README)

In [ ]:
print("Modelo elegido:", elegido)
print("Imágenes entrenamiento / validación / prueba:", len(df_train), "/", len(df_val), "/", len(df_test))
for nombre in p_test:
    mv, mt = metricas(y_val, p_val[nombre]), metricas(y_test, p_test[nombre])
    print(f"{nombre:22s} val_acc={mv['accuracy']:.4f} | test_acc={mt['accuracy']:.4f} | "
          f"test_precision={mt['precision']:.4f} | test_recall={mt['recall']:.4f} | test_f1={mt['f1']:.4f}")
print("Figuras guardadas en assets/:", sorted(p.name for p in DIR_ASSETS.glob("*.png")))